In [0]:
# Databricks Notebook: Delta Lake Optimization Deep Dive (Expert)
# Fully runnable in Databricks Community Edition

from pyspark.sql import functions as F
import random
from datetime import datetime, timedelta
import time

spark.sql("DROP TABLE IF EXISTS demo_zorder")
# ---------------------------------------------
# 1. Create Sample Dataset
# ---------------------------------------------

rows = []
base_date = datetime(2023, 1, 1)

for i in range(300000):
    rows.append((
        i,
        random.choice(["PL", "DE", "FR", "US"]),
        random.randint(1, 5000),
        base_date + timedelta(days=random.randint(0, 365))
    ))

schema = ["id", "country", "user_id", "event_time"]
df = spark.createDataFrame(rows, schema)

(df.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("country")
 .saveAsTable("demo_zorder"))

In [0]:
# ---------------------------------------------
# Checking no of disinct countries
# --
display(df.select("country").distinct().count())

In [0]:
%sql
-- ---------------------------------------------
-- Checking no of distinct partitions
-- ---------------------------------------------
DESCRIBE DETAIL demo_zorder

In [0]:
assert df.select("country").distinct().count() == spark.sql("DESCRIBE DETAIL demo_zorder").first()["numFiles"]

In [0]:
-- ---------------------------------------------
-- Checking no of distinct partitions
-- ---------------------------------------------
DESCRIBE DETAIL demo_zorder

In [0]:
# ---------------------------------------------
# 2. Fragmentation Simulation (WITH FILE COUNT TRACKING)
# ---------------------------------------------

print("Tracking file growth per iteration...")

for i in range(10):
    df.sample(0.03).write.format("delta").mode("append").saveAsTable("demo_zorder")
    
    file_count = spark.sql("DESCRIBE DETAIL demo_zorder").select("numFiles").collect()[0][0]
    print(f"Iteration {i+1}: total files = {file_count}")

print("Final file count BEFORE OPTIMIZE:")
spark.sql("DESCRIBE DETAIL demo_zorder").select("numFiles").show()

In [0]:
# ---------------------------------------------
# 3. OPTIMIZE (File Compaction)
# ---------------------------------------------

spark.sql("OPTIMIZE demo_zorder")

print("Files AFTER OPTIMIZE:")
spark.sql("DESCRIBE DETAIL demo_zorder").select("numFiles").show()

In [0]:
# ---------------------------------------------
# 4. Z-ORDER Performance + File Access Insight
# ---------------------------------------------

query = "SELECT * FROM demo_zorder WHERE country = 'PL' AND user_id = 42"

# Helper to count files touched
from pyspark.sql.functions import input_file_name

def files_touched(q):
    dfq = spark.sql(q).withColumn("file", input_file_name())
    return dfq.select("file").distinct().count()

# BEFORE Z-ORDER
start = time.time()
spark.sql(query).count()
time_before = time.time() - start
files_before = files_touched(query)

print("Time BEFORE Z-ORDER:", time_before)
print("Files scanned BEFORE Z-ORDER:", files_before)

# Apply Z-ORDER
spark.sql("OPTIMIZE demo_zorder ZORDER BY (user_id)")

# AFTER Z-ORDER
start = time.time()
spark.sql(query).count()
time_after = time.time() - start
files_after = files_touched(query)

print("Time AFTER Z-ORDER:", time_after)
print("Files scanned AFTER Z-ORDER:", files_after)

In [0]:
# ---------------------------------------------
# 5. Inspect Delta Table Metadata (Data Skipping)
# ---------------------------------------------

print("Table details:")
spark.sql("DESCRIBE DETAIL demo_zorder").show(truncate=False)

print("History (operations):")
spark.sql("DESCRIBE HISTORY demo_zorder").show(truncate=False)

# Key idea:
print("""
Each file in Delta has min/max statistics per column.
Query engine skips files where values cannot match filters.
Z-ORDER improves clustering → tighter min/max ranges → more skipping.
""")

In [0]:
# ---------------------------------------------
# 6. Partitioning vs Z-ORDER (Advanced View)
# ---------------------------------------------

print("""
Partitioning (country):
- Eliminates entire folders
- Best for LOW cardinality

Z-ORDER (user_id):
- Optimizes WITHIN partitions
- Best for HIGH cardinality

Rule:
Partition first → Z-ORDER inside partitions
""")

In [0]:
# ---------------------------------------------
# 7. Time Travel + Failure Recovery Scenario
# ---------------------------------------------

# Simulate bad overwrite
spark.sql("DELETE FROM demo_zorder WHERE country = 'PL'")

print("After DELETE:", spark.table("demo_zorder").count())

# Recover previous version
history = spark.sql("DESCRIBE HISTORY demo_zorder").collect()
version_before_delete = history[1][0]  # previous version

recovered_df = spark.read.format("delta").option("versionAsOf", version_before_delete).table("demo_zorder")
print("Recovered count:", recovered_df.count())

In [0]:
# ---------------------------------------------
# 8. VACUUM (Storage vs Time Travel Tradeoff)
# ---------------------------------------------

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# WARNING: This deletes history permanently
spark.sql("VACUUM demo_zorder RETAIN 0 HOURS")

print("VACUUM completed - old versions removed")

In [0]:
# ---------------------------------------------
# 9. Bronze → Silver → Gold Pipeline Simulation
# ---------------------------------------------

spark.sql("DROP TABLE IF EXISTS bronze")
spark.sql("DROP TABLE IF EXISTS silver")
spark.sql("DROP TABLE IF EXISTS gold")

# Bronze (raw data)
df.write.format("delta").mode("overwrite").saveAsTable("bronze")

# Silver (cleaned)
spark.sql("""
CREATE TABLE silver AS
SELECT id, country, user_id, event_time
FROM bronze
WHERE user_id IS NOT NULL
""")

# Gold (aggregated)
spark.sql("""
CREATE TABLE gold AS
SELECT country, COUNT(*) as events
FROM silver
GROUP BY country
""")

# Optimize layers differently
spark.sql("OPTIMIZE silver ZORDER BY (user_id)")
spark.sql("OPTIMIZE gold")

print("Pipeline created: bronze → silver → gold")

In [0]:
# ---------------------------------------------
# 10. Production Strategy (Realistic)
# ---------------------------------------------

print("""
PRODUCTION BEST PRACTICES:

Bronze:
- No OPTIMIZE (write-heavy)

Silver:
- Daily OPTIMIZE
- Z-ORDER on query columns

Gold:
- OPTIMIZE frequently
- Small, highly queried tables

VACUUM:
- Run with 7-30 day retention
- Never 0 hours in production

Z-ORDER:
- Apply only on most used filters
- Avoid too many columns
""")